# **1. Data Reading**

In [39]:
import pandas as pd

In [ ]:
#Reading Dataset
data = pd.read_csv('/content/Carbon_Footprint_Estimation.csv')
data

# **2. Exploratory Data Analysis**

In [ ]:
# Statistical Summary of Dataset
data.describe()

In [ ]:
#Checking the size of the dataset(rows, columns)
print(data.shape)

In [ ]:
#Checking Columns name
print(data.columns.tolist())

In [ ]:
#Check data types of all columns
data.dtypes

In [ ]:
#Checking if there is any Null Value present in the dataset or not
print(data.isnull().values.any()) #True means there is null value(s) in the dataset

In [ ]:
#Get total number of null values in the entire dataset
print(data.isnull().sum().sum())

In [ ]:
#Get the count of null values in each column
print(data.isnull().sum())

In [ ]:
# Values in Vehicle Type Columns
data['Vehicle Type']

In [49]:
# Get the non-null values from vehicle type column
non_null_values = (data['Vehicle Type'].dropna().values)

In [ ]:
print(non_null_values)

In [51]:
# Count the null values in the Vehicle Type column
null_values = data['Vehicle Type'].isnull().sum()

In [ ]:
print(null_values)

In [53]:
import numpy as np

In [54]:
#Randomly using the sample values to fill the nulls
random_fill = np.random.choice(non_null_values, size=null_values, replace=True)

In [55]:
#Assign the present sampled values to the NaN positions
data.loc[data['Vehicle Type'].isnull(), 'Vehicle Type'] = random_fill

In [ ]:
data     #dataset without null value

In [ ]:
# Counting how Many Times Each Non-Null Value is ther in the Dataset
count = data['Vehicle Type'].value_counts()
print(count)

In [ ]:
#Re-Checking if there is any Null Value present in the dataset (data) or not
print(data.isnull().values.any())  # Should return False

# **3. Data Preprocessing**

In [59]:
import ast
# Convert list-like strings to real Python lists
def parse_list_column(col):
    return col.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

In [60]:
data['Recycling'] = parse_list_column(data['Recycling'])
data['Cooking_With'] = parse_list_column(data['Cooking_With'])

In [61]:
# Create binary columns for each unique value in list columns
recycling_options = set(item for sublist in data['Recycling'] for item in sublist)
cooking_options = set(item for sublist in data['Cooking_With'] for item in sublist)

In [62]:
for option in recycling_options:
    data[f'Recycling_{option}'] = data['Recycling'].apply(lambda x: int(option in x))

for option in cooking_options:
    data[f'Cooking_{option}'] = data['Cooking_With'].apply(lambda x: int(option in x))

In [63]:
# One-hot encode categorical variables
categorical_columns = ['Body Type','Sex','Diet','How Often Shower','Heating Energy Source','Transport','Vehicle Type','Social Activity','Frequency of Traveling by Air','Waste Bag Size','Energy efficiency']

In [ ]:
data = pd.get_dummies(data, columns=categorical_columns, drop_first=False, dtype = int )
data

In [65]:
data.drop(['Recycling', 'Cooking_With'], axis=1, inplace=True)

In [66]:
# Define features and target
X = data.drop('Carbon Emission (kg/month)', axis=1)   #feature
y = data['Carbon Emission (kg/month)']   #label or target

In [67]:
import sklearn
from sklearn.model_selection import train_test_split
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.columns.tolist)  # List of all feature names in the model
len(X_train.columns)      # Total number of features in the model

# **4.Training Model**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
# Train Random Forest model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
print(model.predict(X_test))    #The test dataset containing input features (not including target/labels)

In [ ]:
data['Predicted_Carbon_Footprint'] = model.predict(data.drop('Carbon Emission (kg/month)', axis=1))     # Prediction for each row
data

# **5. Evaluation Matrix**

In [ ]:
# Check scores
train_score = model.score(X_train, y_train)*100
test_score = model.score(X_test, y_test)*100
print("Model Training Score :",train_score)
print("Model Testing Score :",test_score)

In [73]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Predict and evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

In [ ]:
# Print evaluation metrics
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R^2 Score: {r2:.2f}")

# **6. Model Creation & Saving**

In [75]:
import pickle
# Replace `model` with your actual model variable
with open("carbon_footprint_estimation_model.pkl", "wb") as f:
    pickle.dump(model, f)